# Stain normalisation: does it help our model?

**The question:** with everything else held fixed, does normalising the tiles'
stain before encoding raise the score across hospitals, in both directions,
beyond seed and slide noise?

One change only. Each feature set below is the baseline's pipeline (same tiles
from `get_tiles`, same frozen UNI2-h, same fp16) with one stain method applied
to the tiles between tiling and encoding (`src/stain.py`, targets from
notebook 13). Each is packed and verified exactly as in Stage 11, then trained
and scored exactly as the baseline in notebook 12: gated ABMIL with fixed
settings; mixed on the Stage 4 folds; Radboud to Karolinska and Karolinska to
Radboud, every slide of one hospital to every slide of the other, five seeds
each.

| Version | Stain method |
|---|---|
| `uni2h-L1-224-t144-hsv-v1` | none (the baseline, notebook 12) |
| `uni2h-L1-224-t144-hsv-reinhard_tissue-v1` | reinhard_tissue |
| `uni2h-L1-224-t144-hsv-macenko-v1` | macenko |
| `uni2h-L1-224-t144-hsv-vahadane-v1` | vahadane |
| `uni2h-L1-224-t144-hsv-retinex-v1` | retinex |

**Decision rule.** A method helps only if, in both cross-site directions, the
95% paired bootstrap interval of its QWK minus the baseline's (slides
resampled, five seeds averaged) lies above zero, and on mixed the interval does
not lie below zero. A method that helps in one direction only is not a win:
the targets come from Radboud slides, which favours normalisation when Radboud
is the test hospital.

Extraction resumes: a slide's file is skipped if it exists, so the notebook can
be stopped and re-run. Expect about 3.5 to 4 hours per method on the RTX 4060,
then about 35 minutes per method to pack, verify and train.

## Preamble

In [1]:
"""Make the project's src/ folder importable.

Locally it sits one folder up from notebooks/. On Kaggle it arrives as an
attached dataset, so every attached dataset is checked for a src/ folder, and
the notebook stops with a clear message if there is none.
"""

import glob
import os
import sys

SRC_ROOT = next(
    (
        root
        for root in [os.path.abspath("..")] + sorted(glob.glob("/kaggle/input/*"))
        if os.path.isfile(os.path.join(root, "src", "tiling.py"))
    ),
    None,
)
if SRC_ROOT is None:
    raise FileNotFoundError("src/ not found; on Kaggle, attach the project's src folder as a dataset")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

In [2]:
"""Imports.

os, tempfile, time    paths, the scratch folder for the extraction check, timing.
numpy, pandas, torch  arrays, tables; freeing GPU memory between steps.
tqdm                  progress.
src.encode            the offline encoder.
src.extract           extraction with the stain step, and the manifest.
src.features          check, pack and verify (Stage 11), and per-slide reads.
src.mil               the baseline's model, protocols, scores and paired bootstraps.
src.paths             folds.csv and the weights, locally and on Kaggle.
src.stain             the methods' targets.
"""

import os
import tempfile
import time

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from src.encode import (
    load_encoder_offline,  # builds UNI2-h and loads the weights file strictly, without the network
    prepare,  # eval mode, fp16 on the GPU
)
from src.extract import (
    extract_set,  # extracts every slide not yet saved, CPU workers tiling and normalising while the GPU encodes; returns the failed slides
    write_extraction_manifest,  # the set's manifest from its per-slide files, with the stain section
)
from src.features import (
    check_complete,  # stops unless the per-slide folder holds exactly the expected slides and the manifest says masks_used: false
    open_packed,  # opens a packed set, features memory-mapped
    pack,  # packs a complete set into features.npy, index.parquet, tiles.parquet
    predictions_file,  # where a model's predictions on a set are saved
    read_slide,  # one per-slide file with its format checked
    verify,  # the nine Stage 11 checks
)
from src.mil import (
    CROSS_SITE,  # the two cross-site directions: name -> (training hospital, test hospital)
    HOSPITALS,  # "radboud", "karolinska"
    cross_site,  # every slide of one hospital to every slide of the other, once per seed
    cross_validate,  # the mixed protocol on the Stage 4 folds
    grade_distribution,  # true against predicted grade shares
    load_bags,  # a packed set's features into RAM
    paired_bootstrap_seeds,  # 95% interval of a QWK difference, slides resampled, seeds averaged
    save_predictions,  # predictions to <version>/predictions/<name>.parquet
    seed_summary,  # QWK per seed and their mean and spread
)
from src.paths import (
    DATA_DIR,  # folds.csv lives here locally
    find_input,  # finds folds.csv
    weights_path,  # the UNI2-h weights file, locally or the private Kaggle dataset
)
from src.stain import (
    TARGETS,  # each method's target; the run stops if any is empty
)

In [3]:
"""Settings.

1. METHODS are the stain methods to run, in order; VERSIONS their feature sets.
2. BASELINE is the version notebook 12 trained; its saved predictions are what
   every method is compared with.
3. EXCLUDED is Stage 11's: the empty slide has no tiles in any set.
4. OVERWRITE repacks a set that is already packed.
5. CHECK_SLIDES are the reproduction test's five slides, used to prove the
   extraction code reproduces the stored tissue set before the long run.
6. The run stops here if a method has no target in src/stain.py, or if the
   baseline's predictions are missing (run notebook 12 first).
"""

METHODS = ["reinhard_tissue", "macenko", "vahadane", "retinex"]
VERSIONS = {method: f"uni2h-L1-224-t144-hsv-{method}-v1" for method in METHODS}
BASELINE = "uni2h-L1-224-t144-hsv-v1"
PROTOCOLS = ["mixed", *CROSS_SITE]
EXCLUDED = {
    "3790f55cad63053e956fb73027179707": "empty file: every pixel white at every level, zero tiles (FINDINGS, Stage 10)",
}
OVERWRITE = False
CHECK_SLIDES = [
    "1836f6539ccc9e37d426603cc4526f8b",
    "4726ffb66a6516baff6de98df2b5a26b",
    "0240270d611683829ea5bee2a8c52eb0",
    "03b113d63b06f6740d82b4cfa1ee4a75",
    "3790f55cad63053e956fb73027179707",
]

folds = pd.read_csv(find_input("folds.csv", local_dir=DATA_DIR)).set_index("slide_id")
expected_ids = sorted(folds.index)
missing_targets = [m for m in METHODS if TARGETS[m] is None]
if missing_targets:
    raise ValueError(f"no target in src/stain.py for {missing_targets}; run notebook 13 first")
missing_baseline = [p for p in PROTOCOLS if not os.path.exists(predictions_file(BASELINE, f"abmil-{p}"))]
if missing_baseline:
    raise FileNotFoundError(f"baseline predictions missing for {missing_baseline}; run notebook 12 first")
print(f"{len(expected_ids):,} slides; methods {METHODS}")

folds.csv: /home/aztrek/projects/Prostate-Cancer-Grading/data/folds.csv
10,616 slides; methods ['reinhard_tissue', 'macenko', 'vahadane', 'retinex']


## Does the extraction code reproduce the stored set?

With no stain method, extraction must give exactly the stored tissue features:
same tiles, same coordinates, same fp16 features on this GPU. Checked on the
reproduction test's five slides, written to a scratch folder. The run stops
here if anything differs.

In [4]:
"""Extract the five check slides with no stain method into a scratch folder; compare with the stored set bit for bit."""

encoder = prepare(load_encoder_offline(weights_path()))
scratch = tempfile.mkdtemp()
failed = extract_set(BASELINE, CHECK_SLIDES, encoder, method=None, features_dir=scratch, num_workers=2)
if failed:
    raise RuntimeError(f"extraction failed on {failed}")
for slide_id in CHECK_SLIDES:
    new, stored = read_slide(BASELINE, slide_id, scratch), read_slide(BASELINE, slide_id)
    if not all(a.shape == b.shape and np.array_equal(a, b) for a, b in zip(new, stored)):
        raise RuntimeError(f"{slide_id}: extraction does not reproduce the stored set")
    print(f"{slide_id}: {len(new[0])} tiles, identical")

uni2h-L1-224-t144-hsv-v1: 0 already done, 5 to do
uni2h-L1-224-t144-hsv-v1: 0 slides failed
1836f6539ccc9e37d426603cc4526f8b: 144 tiles, identical
4726ffb66a6516baff6de98df2b5a26b: 144 tiles, identical
0240270d611683829ea5bee2a8c52eb0: 59 tiles, identical
03b113d63b06f6740d82b4cfa1ee4a75: 70 tiles, identical
3790f55cad63053e956fb73027179707: 0 tiles, identical


## Extract

One method after another, over all 10,616 slides, then its manifest. The
encoder stays loaded. A failed slide stops the run with its error; re-running
resumes.

In [5]:
"""Extract every method's set (resuming), then write its manifest; stop if any slide failed."""

for method, version in VERSIONS.items():
    started = time.perf_counter()
    failed = extract_set(version, expected_ids, encoder, method=method)
    if failed:
        raise RuntimeError(f"{version}: {len(failed)} slides failed: {failed}")
    manifest = write_extraction_manifest(version, expected_ids, method=method, progress=tqdm)
    print(f"{version}: {manifest['tiles']:,} tiles, zero-tile {manifest['zero_tile_slides']}, "
          f"stain not fitted on {len(manifest['stain']['not_fitted_slides'])} slides "
          f"{manifest['stain']['not_fitted_slides'][:10]}, {(time.perf_counter() - started) / 3600:.2f} h")
del encoder
torch.cuda.empty_cache()

uni2h-L1-224-t144-hsv-reinhard_tissue-v1: 0 already done, 10616 to do


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f4b4d5713a0>
Traceback (most recent call last):
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f4b4d5713a0>  File "/home/aztrek/projects/Prostate-Cancer-Grading/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__

    Traceback (most recent call last):
self._shutdown_workers()  File "/home/aztrek/projects/Prostate-Cancer-Grading/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__

      File "/home/aztrek/projects/Prostate-Cancer-Grading/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
self._shutdown_workers()    
if w.is_alive():  File "/home/aztrek/projects/Prostate-Cancer-Grading/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers

     if w.is_alive(): Exception ignored in: 
 <function _MultiProcessingDat

uni2h-L1-224-t144-hsv-reinhard_tissue-v1: 0 slides failed
uni2h-L1-224-t144-hsv-reinhard_tissue-v1: 691,236 tiles, zero-tile ['3790f55cad63053e956fb73027179707'], stain not fitted on 0 slides [], 3.04 h
uni2h-L1-224-t144-hsv-macenko-v1: 0 already done, 10616 to do
uni2h-L1-224-t144-hsv-macenko-v1: 0 slides failed
uni2h-L1-224-t144-hsv-macenko-v1: 691,236 tiles, zero-tile ['3790f55cad63053e956fb73027179707'], stain not fitted on 0 slides [], 2.91 h
uni2h-L1-224-t144-hsv-vahadane-v1: 0 already done, 10616 to do


uni2h-L1-224-t144-hsv-retinex-v1: 0 slides failed
uni2h-L1-224-t144-hsv-retinex-v1: 691,236 tiles, zero-tile ['3790f55cad63053e956fb73027179707'], stain not fitted on 0 slides [], 2.90 h


## Pack and verify

Stage 11's functions, unchanged: complete, packed into one memory-mapped array,
and all nine checks including the round trip.

In [6]:
"""Check, pack and verify each set."""

for version in VERSIONS.values():
    check_complete(version, expected_ids)
    summary = pack(version, expected_ids, EXCLUDED, overwrite=OVERWRITE)
    report = verify(version, expected_ids)
    print(f"{version}: packed {summary['slides']:,} slides, {summary['tiles']:,} tiles in {summary['seconds']:.0f} s; "
          f"{len(report)} checks passed")

uni2h-L1-224-t144-hsv-reinhard_tissue-v1: packed 10,615 slides, 691,236 tiles in 134 s; 9 checks passed
uni2h-L1-224-t144-hsv-macenko-v1: packed 10,615 slides, 691,236 tiles in 16 s; 9 checks passed
uni2h-L1-224-t144-hsv-vahadane-v1: packed 10,615 slides, 691,236 tiles in 16 s; 9 checks passed
uni2h-L1-224-t144-hsv-retinex-v1: packed 10,615 slides, 691,236 tiles in 16 s; 9 checks passed


## Train and predict

The baseline's model and protocols (`src/mil.py`), one set in memory at a time.
Predictions are saved next to each set, as in notebook 12.

In [7]:
"""Train and predict each set: mixed on the folds (seed 0), each cross-site direction with five seeds; save."""

for method, version in VERSIONS.items():
    packed = open_packed(version)
    labels = folds.loc[packed.index["slide_id"]]
    bags = load_bags(packed)
    started = time.perf_counter()
    predicted = {"mixed": cross_validate(bags, labels, "mixed", progress=tqdm)}
    for name, (train_hospital, test_hospital) in CROSS_SITE.items():
        predicted[name] = cross_site(bags, labels, train_hospital, test_hospital, progress=tqdm)
    for protocol, values in predicted.items():
        save_predictions(version, f"abmil-{protocol}", labels, values)
    print(f"{method}: trained in {(time.perf_counter() - started) / 60:.1f} min")
    del bags, packed

reinhard_tissue: trained in 23.2 min
macenko: trained in 23.3 min
vahadane: trained in 23.2 min
retinex: trained in 23.2 min


## Score against the baseline

Per method and protocol, on exactly the baseline's slides: the baseline's QWK,
the method's, the difference, and its 95% paired bootstrap interval (slides
resampled 2,000 times; for cross-site, each run's five seeds averaged on every
draw). Mixed is also split by hospital.

In [8]:
"""Compare every method with the baseline, slide for slide.

compare_to_baseline(method, protocol) returns one row per slide group: the
baseline's and the method's QWK (seed means for cross-site), the difference,
and its paired bootstrap interval.
"""


def prediction_columns(table):
    """The prediction columns of a saved predictions table: "predicted", or seed_0 to seed_4."""
    return [c for c in table.columns if c == "predicted" or c.startswith("seed_")]


def compare_to_baseline(method, protocol):
    base = pd.read_parquet(predictions_file(BASELINE, f"abmil-{protocol}"))
    run = pd.read_parquet(predictions_file(VERSIONS[method], f"abmil-{protocol}")).loc[base.index]
    groups = {"all": base.index}
    if protocol == "mixed":
        groups.update({h: base.index[base["provider"] == h] for h in HOSPITALS})
    rows = []
    for group, ids in groups.items():
        truth = base.loc[ids, "isup_grade"]
        before = base.loc[ids, prediction_columns(base)]
        after = run.loc[ids, prediction_columns(run)]
        low, high = paired_bootstrap_seeds(truth, after, before)
        rows.append({"method": method, "protocol": protocol, "slides": group, "n": len(ids),
                     "baseline": seed_summary(truth, before)["mean"], "normalised": seed_summary(truth, after)["mean"],
                     "seed std": seed_summary(truth, after)["std"], "95% low": low, "95% high": high})
    return rows


comparison = pd.DataFrame([row for m in METHODS for p in PROTOCOLS for row in compare_to_baseline(m, p)])
comparison["difference"] = comparison["normalised"] - comparison["baseline"]
print(comparison.round(4).to_string(index=False))

         method              protocol     slides     n  baseline  normalised  seed std  95% low  95% high  difference
reinhard_tissue                 mixed        all 10615    0.8643      0.8616    0.0000  -0.0092    0.0038     -0.0027
reinhard_tissue                 mixed    radboud  5160    0.8240      0.8248    0.0000  -0.0095    0.0111      0.0009
reinhard_tissue                 mixed karolinska  5455    0.8807      0.8725    0.0000  -0.0179    0.0014     -0.0082
reinhard_tissue radboud_to_karolinska        all  5455    0.6316      0.5887    0.0260  -0.0594   -0.0265     -0.0429
reinhard_tissue karolinska_to_radboud        all  5160    0.7800      0.7559    0.0067  -0.0338   -0.0144     -0.0241
        macenko                 mixed        all 10615    0.8643      0.8601    0.0000  -0.0110    0.0030     -0.0042
        macenko                 mixed    radboud  5160    0.8240      0.8222    0.0000  -0.0130    0.0095     -0.0018
        macenko                 mixed karolinska  5455  

## Predicted grades

The share of slides predicted at each grade, five seeds pooled, for the
baseline and every method, against the true share. A cross-site gain that is
only a shift towards lower or higher grades shows up here.

In [9]:
"""Predicted-grade distribution per cross-site direction: true, baseline, then each method."""

for protocol in CROSS_SITE:
    base = pd.read_parquet(predictions_file(BASELINE, f"abmil-{protocol}"))
    table = {"true": grade_distribution(base["isup_grade"], base[prediction_columns(base)])["true"],
             "baseline": grade_distribution(base["isup_grade"], base[prediction_columns(base)])["predicted"]}
    for method, version in VERSIONS.items():
        run = pd.read_parquet(predictions_file(version, f"abmil-{protocol}")).loc[base.index]
        table[method] = grade_distribution(run["isup_grade"], run[prediction_columns(run)])["predicted"]
    print(f"\n{protocol}: share of slides per grade")
    print(pd.DataFrame(table).T.round(3).to_string())


radboud_to_karolinska: share of slides per grade
grade                0      1      2      3      4      5
true             0.353  0.333  0.122  0.058  0.088  0.046
baseline         0.386  0.253  0.055  0.162  0.094  0.051
reinhard_tissue  0.345  0.194  0.154  0.177  0.060  0.070
macenko          0.401  0.261  0.064  0.174  0.053  0.047
vahadane         0.398  0.240  0.070  0.181  0.061  0.050
retinex          0.328  0.248  0.135  0.148  0.073  0.067

karolinska_to_radboud: share of slides per grade
grade                0      1      2      3      4      5
true             0.187  0.165  0.131  0.179  0.149  0.189
baseline         0.207  0.151  0.139  0.096  0.296  0.112
reinhard_tissue  0.167  0.148  0.179  0.060  0.233  0.213
macenko          0.208  0.166  0.148  0.061  0.301  0.117
vahadane         0.200  0.166  0.157  0.067  0.296  0.113
retinex          0.198  0.151  0.126  0.144  0.194  0.186


## The decision rule

Applied per method: both cross-site intervals above zero, and the mixed
interval (all slides) not below zero.

In [10]:
"""Apply the decision rule to every method and print the verdict with the numbers behind it."""

for method in METHODS:
    rows = comparison[(comparison["method"] == method) & (comparison["slides"] == "all")].set_index("protocol")
    cross_ok = all(rows.loc[p, "95% low"] > 0 for p in CROSS_SITE)
    mixed_ok = rows.loc["mixed", "95% high"] >= 0
    verdict = "helps" if cross_ok and mixed_ok else "does not help"
    details = "; ".join(f"{p} {rows.loc[p, 'difference']:+.3f} [{rows.loc[p, '95% low']:+.3f}, {rows.loc[p, '95% high']:+.3f}]"
                        for p in PROTOCOLS)
    print(f"{method:16s} {verdict:14s} {details}")

reinhard_tissue  does not help  mixed -0.003 [-0.009, +0.004]; radboud_to_karolinska -0.043 [-0.059, -0.027]; karolinska_to_radboud -0.024 [-0.034, -0.014]
macenko          does not help  mixed -0.004 [-0.011, +0.003]; radboud_to_karolinska +0.047 [+0.030, +0.063]; karolinska_to_radboud -0.009 [-0.017, -0.002]
vahadane         does not help  mixed -0.008 [-0.015, -0.001]; radboud_to_karolinska +0.016 [-0.001, +0.032]; karolinska_to_radboud -0.007 [-0.014, +0.001]
retinex          does not help  mixed +0.002 [-0.005, +0.008]; radboud_to_karolinska -0.022 [-0.038, -0.006]; karolinska_to_radboud -0.004 [-0.011, +0.004]


## Result

Run 2026-11-17 to 18 on the RTX 4060 PC; extraction and packing at commit
`8df64a2`, recorded in every manifest.

- **Extraction check:** with no stain method, the five reproduction-test slides
  came out bit for bit identical to the stored tissue set.
- **Four sets, each complete:** 10,616 slides, 691,236 tiles, the empty slide
  the only zero-tile one, stain fitted on every other slide, about 2.9 to 3.0 h
  of extraction each (1.0 s per slide, GPU-bound). All packed and passing the
  nine checks. 23 minutes of training per set.
- The "Exception ignored ... can only test a child process" messages during
  extraction are harmless: worker processes of a later extraction inherit the
  previous one's data-loader handle and fail to shut it down. Every slide was
  extracted and verified.
- scikit-learn printed an expected "Objective did not converge" warning
  573,869 times during Vahadane extraction (dictionary learning stops at 100
  iterations; the stain vectors agree to cosine 1.000 with a four times larger
  fit, FINDINGS). They are removed from the saved output above, which would
  otherwise be 230 MB; `src/stain.py` now silences that one warning.

**No method helps by the decision rule.** Difference from the baseline with its
95% paired bootstrap interval:

| Method | Mixed | Radboud to Karolinska | Karolinska to Radboud |
|---|---|---|---|
| Reinhard, tissue only | -0.003 [-0.009, +0.004] | -0.043 [-0.059, -0.027] | -0.024 [-0.034, -0.014] |
| Macenko | -0.004 [-0.011, +0.003] | **+0.047 [+0.030, +0.063]** | -0.009 [-0.017, -0.002] |
| Vahadane | -0.008 [-0.015, -0.001] | +0.016 [-0.001, +0.032] | -0.007 [-0.014, +0.001] |
| Retinex | +0.002 [-0.005, +0.008] | -0.022 [-0.038, -0.006] | -0.004 [-0.011, +0.004] |

Baseline: 0.864 mixed, 0.632 Radboud to Karolinska, 0.780 Karolinska to
Radboud. Macenko is the only clear gain, in one direction, at a small cost in
the other. The methods that match the hospitals' colours best (tissue-only
Reinhard, Retinex) are the ones that lose. Details and open questions:
FINDINGS, "Stain normalisation ablation".